# Compare ConSim Specifications and Interpretations

Stage 1 asks whether we correctly reconstructed the old ConSim setup and whether the newer prompt formats improve on it. This notebook keeps that analysis explicit and small.

The previous working notebook was archived to `notebooks/old/4_compare_consim.ipynb` for reference.

What this notebook does:

1. Compare two ConSim specifications selected at the top of the notebook.
2. Compare two concept interpretations selected at the top of the notebook.
3. Plot score distributions for the selected specifications.
4. Reproduce old-paper-style pairwise matrices on `old_consim` rows only.

Reusable visualization code lives in `utils/plot.py`. This notebook should mainly prepare data and explain decisions.


## 1. Parameters

Change these values to choose the comparison. Keep everything else deterministic and explicit.


In [ ]:
# Judge model file stem. The notebook first looks for data/consim_{MODEL_FILE}_v2.csv.
MODEL_FILE = "Qwen_Qwen3.5-9B"

# Main specification comparison.
SPEC_A = "old_consim"
SPEC_B = "new_consim"

# Main interpretation comparison.
INTERP_A = "topk"
INTERP_B = "llm"

# Optional filters. Use None to keep all values present in the score file.
DATASETS = ["IMDB", "RT", "BIOS", "E"]             # e.g. ["RT", "BIOS"]
PROMPT_TYPES = None         # e.g. ["C1", "C2", "B1", "B2"]
CLASSES_SUBSETS = None      # e.g. ["[0, 1]"]
METHODS = None              # e.g. ["SemiNMF", "ICA"]

# Bucket convention used throughout this repo: 50 seeds -> 5 groups of 10.
SEEDS_PER_BUCKET = 10
EXPECTED_SEEDS = 50

# Paper exports. Set EXPORT_FIGURES=False while iterating interactively.
EXPORT_FIGURES = False


## 2. Imports and Paths

Only plotting style is abstracted. Filtering and joins stay visible in this notebook for auditability.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from utils.plot import (
    DEFAULT_BASELINE_FOR,
    plot_accuracies_violins,
    plot_difference_bars,
    plot_pairwise_comparison_matrices,
)

DATA_DIR = REPO_ROOT / "data"
EXPORT_DIR = REPO_ROOT / "LaTeX-Simulatability-Shortcut" / "plots"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)


## 3. Load Scores

Prefer v2 score files because they contain parser diagnostics. Fall back to the legacy file only if v2 does not exist.


In [ ]:
CSV_PATH = DATA_DIR / f"consim_{MODEL_FILE}_v2.csv"
assert CSV_PATH.exists(), f"No score CSV found at {CSV_PATH}"

df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df):,} rows from {CSV_PATH.relative_to(REPO_ROOT)}")
print(f"specifications: {sorted(df['specification'].dropna().unique())}")
print(f"interpretations: {sorted(map(str, df['interpretation'].dropna().unique()))}")
print(f"datasets: {sorted(df['dataset'].dropna().unique())}")
print(f"prompt_types: {sorted(df['prompt_type'].dropna().unique())}")
df.head()


## 4. Shared Data Helpers

These helpers only prepare data. They do not draw figures.


In [ ]:
KEY_COLS = [
    "dataset",
    "model",
    "classes_subset",
    "seed",
    "method",
    "nb_concepts",
    "interpretation",
    "prompt_type",
]

EXPLANATION_PROMPT_TYPES = set(DEFAULT_BASELINE_FOR)
BASELINE_PROMPT_TYPES = set(DEFAULT_BASELINE_FOR.values())
ROW_KEY_COLS = KEY_COLS + ["specification"]
BASELINE_KEY_COLS = ["dataset", "model", "classes_subset", "seed", "prompt_type", "specification"]

def apply_filter(frame: pd.DataFrame, column: str, values) -> pd.DataFrame:
    """Apply an exact-match optional filter and print missing requested values."""
    if values is None:
        return frame
    values = list(values)
    missing = set(values) - set(frame[column].dropna().unique())
    if missing:
        print(f"warning: {column} values not present before filtering: {sorted(missing)}")
    return frame[frame[column].isin(values)].copy()


def base_filter(frame: pd.DataFrame) -> pd.DataFrame:
    """Apply the user-facing filters from the parameter cell."""
    out = frame.copy()
    out = apply_filter(out, "dataset", DATASETS)
    out = apply_filter(out, "prompt_type", PROMPT_TYPES)
    out = apply_filter(out, "classes_subset", CLASSES_SUBSETS)
    out = apply_filter(out, "method", METHODS)
    return out


def canonicalize_baselines_and_validate(frame: pd.DataFrame) -> pd.DataFrame:
    """Mean duplicated baselines and reject duplicated non-baseline rows.

    Baseline prompts are shared across methods. They can therefore appear
    multiple times with different method-specific metadata. For analysis they
    have one semantic key: dataset/model/classes_subset/seed/prompt_type/
    specification. We average their scores on that key.

    Non-baseline rows should already be unique on the full row key. If they
    are duplicated, stop instead of silently averaging a real experiment.
    """
    baseline_mask = (frame["method"] == "baseline") | frame["prompt_type"].isin(BASELINE_PROMPT_TYPES)
    baselines = frame[baseline_mask].copy()
    non_baselines = frame[~baseline_mask].copy()

    duplicate_non_baselines = (
        non_baselines.groupby(ROW_KEY_COLS, dropna=False)
        .size()
        .reset_index(name="n")
        .query("n > 1")
    )
    if not duplicate_non_baselines.empty:
        display(duplicate_non_baselines.head(20))
        raise ValueError("Duplicated non-baseline score rows found. Inspect the displayed keys before continuing.")

    if baselines.empty:
        return non_baselines

    before = len(baselines)
    baseline_scores = baselines.groupby(BASELINE_KEY_COLS, dropna=False)["score"].mean().reset_index()

    # Rebuild baseline rows with canonical metadata. Other columns are not
    # used in this notebook, so keep them as NA when present.
    collapsed = pd.DataFrame(columns=frame.columns)
    for column in baseline_scores.columns:
        collapsed[column] = baseline_scores[column]
    collapsed["method"] = "baseline"
    collapsed["nb_concepts"] = np.nan
    collapsed["interpretation"] = np.nan

    after = len(collapsed)
    if before != after:
        print(f"Averaged duplicated baseline rows: {before:,} -> {after:,}")

    return pd.concat([non_baselines, collapsed], ignore_index=True)


def common_value_table(frame: pd.DataFrame, variable_col: str, value_a, value_b, key_cols: list[str]) -> pd.DataFrame:
    """Return rows whose key exists for both selected values.

    The output has columns `value_a`, `value_b`, and `diff = value_b - value_a`.
    Exact duplicate score rows are averaged first to make rescoring deterministic.
    """
    small = frame[frame[variable_col].isin([value_a, value_b])].copy()
    # This groupby is a deterministic safeguard after baseline canonicalization.
    # Non-baseline duplicates have already been rejected above.
    grouped = small.groupby(key_cols + [variable_col], dropna=False)["score"].mean().reset_index()
    wide = grouped.pivot_table(index=key_cols, columns=variable_col, values="score", aggfunc="mean")
    wide = wide.dropna(subset=[value_a, value_b]).reset_index()
    wide["diff"] = wide[value_b] - wide[value_a]
    return wide


def bucket_difference_stats(wide: pd.DataFrame, value_a, value_b, group_cols: list[str]) -> pd.DataFrame:
    """Aggregate paired differences into 10-seed buckets.

    We first average all rows within a seed for the current group. Then each
    contiguous block of `SEEDS_PER_BUCKET` seeds gives one bucket mean. Error
    bars are the standard deviation across bucket differences.
    """
    def _one_group(group: pd.DataFrame) -> pd.Series:
        per_seed = group.groupby("seed", as_index=False)[[value_a, value_b]].mean().sort_values("seed")
        n_seeds = len(per_seed)
        if n_seeds != EXPECTED_SEEDS:
            print(f"warning: group={group.name!r} has {n_seeds} seeds, expected {EXPECTED_SEEDS}")
        n_buckets = n_seeds // SEEDS_PER_BUCKET
        if n_buckets == 0:
            return pd.Series({"mean_diff": np.nan, "std_diff": np.nan, "n_buckets": 0, "n_seeds": n_seeds})
        vals_a = per_seed[value_a].to_numpy()[: n_buckets * SEEDS_PER_BUCKET]
        vals_b = per_seed[value_b].to_numpy()[: n_buckets * SEEDS_PER_BUCKET]
        bucket_a = vals_a.reshape(n_buckets, SEEDS_PER_BUCKET).mean(axis=1)
        bucket_b = vals_b.reshape(n_buckets, SEEDS_PER_BUCKET).mean(axis=1)
        diffs = bucket_b - bucket_a
        return pd.Series({
            "mean_diff": float(diffs.mean()),
            "std_diff": float(diffs.std(ddof=1)) if n_buckets > 1 else 0.0,
            "n_buckets": int(n_buckets),
            "n_seeds": int(n_seeds),
        })

    return (
        wide.groupby(group_cols, dropna=False)
        .apply(_one_group, include_groups=False)
        .reset_index()
        .sort_values("mean_diff", ascending=False)
        .reset_index(drop=True)
    )


df_f = canonicalize_baselines_and_validate(base_filter(df))
print(f"Rows after base filters: {len(df_f):,}")


## 5. Specification Comparison

This asks: holding dataset, seed, method, interpretation, and prompt type fixed, does `SPEC_B` score higher than `SPEC_A`?


In [ ]:
spec_wide = common_value_table(
    df_f,
    variable_col="specification",
    value_a=SPEC_A,
    value_b=SPEC_B,
    key_cols=KEY_COLS,
)
print(f"Common keys for {SPEC_A!r} vs {SPEC_B!r}: {len(spec_wide):,}")
spec_wide.head()


In [ ]:
spec_stats = bucket_difference_stats(
    spec_wide,
    value_a=SPEC_A,
    value_b=SPEC_B,
    group_cols=["dataset", "classes_subset", "prompt_type"],
)
# Bar panels must keep dataset and classes_subset together. Otherwise,
# binary datasets such as IMDB and RT get mixed because they share the same
# classes_subset string.
spec_stats["dataset_classes_subset"] = spec_stats["dataset"] + " | " + spec_stats["classes_subset"].astype(str)

fig, axes = plot_difference_bars(
    spec_stats,
    panel_col="dataset_classes_subset",
    ylabel=f"{SPEC_B} - {SPEC_A}",
    title=(
        f"Specification comparison: {SPEC_B} - {SPEC_A} | judge={MODEL_FILE}\n"
        f"error bars: std over {SEEDS_PER_BUCKET}-seed buckets"
    ),
    save_dir=EXPORT_DIR if EXPORT_FIGURES else None,
    file_name=f"conSim_spec_{SPEC_B}_minus_{SPEC_A}.pdf" if EXPORT_FIGURES else None,
)
plt.show()
spec_stats


## 6. Interpretation Comparison

This asks: holding specification, dataset, seed, method, and prompt type fixed, does `INTERP_B` score higher than `INTERP_A`? Baselines are excluded because they do not contain explanations.


In [ ]:
interp_source = df_f[df_f["prompt_type"].isin(EXPLANATION_PROMPT_TYPES)].copy()
interp_source = interp_source[interp_source["method"] != "baseline"]

interp_key_cols = [col for col in KEY_COLS + ["specification"] if col != "interpretation"]
interp_wide = common_value_table(
    interp_source,
    variable_col="interpretation",
    value_a=INTERP_A,
    value_b=INTERP_B,
    key_cols=interp_key_cols,
)
print(f"Common keys for {INTERP_A!r} vs {INTERP_B!r}: {len(interp_wide):,}")
interp_wide.head()


In [ ]:
interp_stats = bucket_difference_stats(
    interp_wide,
    value_a=INTERP_A,
    value_b=INTERP_B,
    group_cols=["specification", "dataset", "classes_subset", "prompt_type"],
)
# Same rule as above: never pool different datasets just because they have
# the same classes_subset representation.
interp_stats["dataset_classes_subset"] = interp_stats["dataset"] + " | " + interp_stats["classes_subset"].astype(str)

for specification, sub_stats in interp_stats.groupby("specification", dropna=False):
    fig, axes = plot_difference_bars(
        sub_stats,
        panel_col="dataset_classes_subset",
        ylabel=f"{INTERP_B} - {INTERP_A}",
        title=(
            f"Interpretation comparison under {specification}: {INTERP_B} - {INTERP_A} | judge={MODEL_FILE}\n"
            f"error bars: std over {SEEDS_PER_BUCKET}-seed buckets"
        ),
        save_dir=EXPORT_DIR if EXPORT_FIGURES else None,
        file_name=f"conSim_interp_{specification}_{INTERP_B}_minus_{INTERP_A}.pdf" if EXPORT_FIGURES else None,
    )
    plt.show()
interp_stats


## 7. Score Distributions for the Selected Specifications

The bar plots above summarize differences. These violins show the raw score distributions for each `(dataset, classes_subset)` pair, with the two selected specifications split inside each prompt/method slot.


In [ ]:
INDEX_LEVELS = [
    "dataset", "model", "classes_subset", "method", "nb_concepts",
    "interpretation", "seed", "specification", "prompt_type",
]

violin_df = df_f[df_f["specification"].isin([SPEC_A, SPEC_B])].copy()
indexed = violin_df.set_index(INDEX_LEVELS)["score"]

for dataset, classes_subset in sorted(set(zip(violin_df["dataset"], violin_df["classes_subset"]))):
    mask = (
        (indexed.index.get_level_values("dataset") == dataset)
        & (indexed.index.get_level_values("classes_subset") == classes_subset)
    )
    pair_series = indexed[mask]
    if pair_series.empty:
        continue

    pt_level = pair_series.index.get_level_values("prompt_type")
    accuracies = pair_series[pt_level.isin(EXPLANATION_PROMPT_TYPES)]
    baselines = pair_series[pt_level.isin(BASELINE_PROMPT_TYPES)]
    if accuracies.empty:
        continue

    methods = sorted(accuracies.index.get_level_values("method").unique())
    ax = plot_accuracies_violins(
        accuracies,
        baselines,
        baseline_for=DEFAULT_BASELINE_FOR,
        compared_index="method",
        method_order=methods,
        split_index="specification",
        split_order=[SPEC_A, SPEC_B],
        title=f"{dataset} | classes_subset={classes_subset} | {SPEC_A} vs {SPEC_B}",
        save_dir=EXPORT_DIR if EXPORT_FIGURES else None,
        file_name=f"conSim_violins_{dataset}_{str(classes_subset).replace(' ', '')}_{SPEC_A}_vs_{SPEC_B}.pdf" if EXPORT_FIGURES else None,
    )
    plt.show()


## 8. Old-ConSim Pairwise Matrices

This section checks whether the reconstructed `old_consim` behavior produces old-paper-style pairwise rankings. We restrict to old-ConSim rows and compare methods plus a duplicated `NoExplanation` baseline.

For simplicity, the matrix collapses across concept hyperparameters and interpretations after filtering. If a stricter reproduction is needed, set `METHODS`, `PROMPT_TYPES`, or `CLASSES_SUBSETS` above and rerun.


In [ ]:
old_df = df_f[df_f["specification"] == "old_consim"].copy()
matrix_prompt_types = [pt for pt in ["C1", "C2", "C3", "AC1", "AC2", "AC3"] if pt in set(old_df["prompt_type"])]

matrix_parts = []
# Explanation methods keep their method names.
exp = old_df[old_df["prompt_type"].isin(matrix_prompt_types) & (old_df["method"] != "baseline")].copy()
exp["contender"] = exp["method"]
matrix_parts.append(exp)

# Duplicate the matching baseline under every explanation prompt type so it
# pairs on the same dataset/classes_subset/seed/prompt_type cells.
for prompt_type in matrix_prompt_types:
    baseline_prompt = DEFAULT_BASELINE_FOR[prompt_type]
    baseline = old_df[old_df["prompt_type"] == baseline_prompt].copy()
    if baseline.empty:
        continue
    baseline["prompt_type"] = prompt_type
    baseline["contender"] = "NoExplanation"
    matrix_parts.append(baseline)

matrix_df = pd.concat(matrix_parts, ignore_index=True) if matrix_parts else pd.DataFrame()
print(f"Pairwise matrix rows: {len(matrix_df):,}")
print(f"Contenders: {sorted(matrix_df['contender'].dropna().unique()) if not matrix_df.empty else []}")


In [ ]:
if matrix_df.empty:
    print("No old_consim rows available for the pairwise matrix.")
else:
    pairwise_index = ["dataset", "model", "classes_subset", "seed", "prompt_type", "contender"]
    pairwise_scores = matrix_df.set_index(pairwise_index)["score"]
    fig_pct, fig_diff, percentage_matrix, difference_matrix = plot_pairwise_comparison_matrices(
        pairwise_scores,
        compared_index="contender",
        title_prefix=f"old_consim reconstruction | judge={MODEL_FILE}",
        save_dir=EXPORT_DIR if EXPORT_FIGURES else None,
        file_prefix="old_consim_reconstruction" if EXPORT_FIGURES else None,
    )
    plt.show()
    display(percentage_matrix)
    display(difference_matrix)


## 9. Decision Notes

Use this final markdown cell after running the notebook to record the Stage 1 decision:

- Does `simulator_consim` improve over `new_consim` enough to justify regenerating broader prompts?
- Does `llm` interpretation change conclusions compared with `topk`?
- Do the `old_consim` pairwise matrices look consistent with the old ConSim reference paper?
